# Reading the Input

Whether the input is a `.mol`, `.sdf` or a  folder of `.mol` files, each function processes them appropriately. 

The `read_mol` and `read_sdf` functions return a dictionary of structures derived from the files with the name of the file as the key. th eoutput of both ensures every molecule entry is isolated and can be read on its own.

The `is_sdf` returns 1 if it's True and 0 if it's false. Finally, the `split_record` divides a mol structure string into a list of lines.

In [ ]:
#Converts the dictionary into a list of mol files
def read_mol(files):
  mol_list = list(files.values())
  return mol_list

#Checks if it's an sdf file
def is_sdf(files):
    import re
    regex_list = [r".+\.mol", r".+\.sdf"]
    for key in files.keys():
        if re.match(regex_list[1], key):
            return 1
        elif re.match(regex_list[0], key):
            return 0

# Returns list of mol files            
def read_sdf(sdf_file):
    # Split the file content by "$$$$" delimiter
    mol_records = list(sdf_file.values())[0].split('$$$$\n') # Have to convert to a list of lines first
    
    if mol_records and len(mol_records[-1]) < 4: # the second condition returns true if the last list (mol) is notvalid
        mol_records.pop()
    return mol_records

# used to split each mol record in the files list into seperate lines
def split_record(record):
    lines = record.splitlines(keepends=True)
    return lines


# Getting the name

To keep track of and label the molecules, we want to be able to retrieve their names. We imported the `rdkit` and `pubchempy` libraries to do so. 

In [ ]:
from rdkit import Chem 
import pubchempy

To search through the pubchem database via `pubchempy`, we had to convert the mol file into a smiles via `rdkit`.

The `getName` function takes a mol input as a string and returns the corresponding name.

In [ ]:
def getName(mol):
    #Converting to a smile
    mol = Chem.MolFromMolBlock(mol)
    smiles = Chem.MolToSmiles(mol)

    #searching the pubchem database for names that match the smiles provided
    compounds = pubchempy.get_compounds(smiles, namespace='smiles')

    #the first index is the collection of found names
    match = compounds[0]

    # synonyms[0] is the first entry of common names and is usually the most commonly used
    return match.synonyms[0] if match.synonyms else match.iupac_name #if there is no common name, return the IUPAC name

# Create Graphs

## Properties
Before creating the graphs, we need to extract certian data from the mol files. We did so by isolating the mol block and bond block via patterns found in `.mol` files. 

The patterns we used:
- Information on atom and bond count is found on the 4th line in each molecule entry.
- The atom block starts on the 5th line
- The bond block starts directly after the atom block

The `properties` function returns a dictionary of the needed parameters

In [ ]:
# Takes only one mol file as a list of lines as input, creates list with certain values for each mol
def properties(lines):
    # Get values
    counts_line = lines[3] 
    num_atoms = int(counts_line[0:3].strip()) # number of atoms
    num_bonds = int(counts_line[3:6].strip()) # number of bonds

    # Identify start and end of atom block and bond block
    ab_start = 4 
    ab_end = ab_start + num_atoms
    bb_start = ab_end
    bb_end = bb_start + num_bonds
    
    return {"num_atoms": num_atoms, "num_bonds": num_bonds, "ab_start": ab_start, "ab_end": ab_end, "bb_start": bb_start, "bb_end": bb_end}


## Removing Hydrogens for calculation
The loop comprehension statement iterates through each node `n` in the copied graph G_copy, and stores the node `n` in the list if it is a hydrogen. Then, the `.remove_nodes_from` in the networkx library removes a set of nodes from a given list. 

In [ ]:
def remove_hydrogens(G):
    G_copy = G.copy()  # Shallow copy so we don't use up extra memory for no reason

    hydrogen_nodes = [n for n, attr in G_copy.nodes(data=True) if attr.get('atomType') == 'H']
    G_copy.remove_nodes_from(hydrogen_nodes)

    return G_copy

## Graph Functions 
In order to create the final graphs dictionary, we needed to intialize three functions: 

`getEBunch` creates and outputs the edge list of a single molecule entry. The input is the molecule as a list of lines.

`create_graph` creates and outputs a single graph from one molecule entry and its corresponding edge list. In this function we implemented the following:
- Removed all hydrogens
- Removed any ions or counter ions found in the same mol block to avoid including them in final calculations.

`create_graph_dic` utilizes the above functions to create the final dictionary of graphs where each graph is idnetified by its common or IUPAC name.


In [ ]:
import networkx as nx

# Takes only one mol file as a list of lines as input, creates edge list from each mol file
def getEBunch(mol):
    lines = mol
    
    # output described above
    value = properties(lines)

    # Get bond tuples
    bond_lines = lines[value["bb_start"]:value["bb_end"]]
    eBunch = []

    # Each bond line has: atom1 atom2 bondType stereo
    for line in bond_lines:
        atom1 = int(line[0:3].strip())
        atom2 = int(line[3:6].strip())
        bond_type = int(line[6:9].strip())
        stereo = int(line[9:12].strip())
        eBunch.append((atom1, atom2, {'bondType': bond_type, 'bondStereo': stereo}))
    return eBunch

# Returns graph from eBunch and assigns atoms types
def create_graph(eBunch, lines):

    # create graph
    Graph = nx.Graph()
    Graph.add_edges_from(eBunch)

    # Assigning the nodes to atoms
    value = properties(lines)
    atom_lines = lines[value["ab_start"]:value["ab_end"]]

    # for every node in the graph, assign the corresponding atom type from the atom block
    for n in Graph.nodes:
        # access the nth atom (atom line index n-1), and get the atom type stored in the 31-33 column
        Graph.nodes[n]["atomType"] = str(atom_lines[n-1][31:34].strip()) 
    
    # remove any lone ions from the final graph
    isolated = list(nx.isolates(Graph))
    Graph.remove_nodes_from(isolated)

    # Keep only the largest component (removing counter ions) 
    if not nx.is_connected(Graph):
        components = list(nx.connected_components(Graph))
        largest = max(components, key=len)
        Graph = Graph.subgraph(largest).copy()
    return Graph

# Returns dictionary of graphs for every mol structure in list, takes files as input
def create_graph_dic(files):
    Graphs = {}
    # checks if it's an sdf file. If it is, will read as sdf, otherwise will read as mol
    mol_rec = read_sdf(files) if is_sdf(files) else read_mol(files)
    for mol in mol_rec:
        name = getName(mol) # getting common or IUPAC name
        lines = split_record(mol)
        eBunch = getEBunch(lines)
        G = create_graph(eBunch, lines)
        Graphs[name] = G
    return Graphs

# Indices

All our indices are calculated from scratch and the three indices have their corresponding `control` functions in order to validate the output. Note that the final function for every index takes the dictionary of graphs as input and outputs a dictionary of values such that `{"Name of molecule": value}`

## Edge Density
This function calculates the edge density *manually* for each molecular graph by first determining the number of nodes and edges using `len()` on the graph object. These values are then used to compute the density using the following formula:

$$
\text{Edge Density} = \frac{2E}{N(N - 1)}
$$

Where:

- $E$ represents the number of edges in the graph

- $N$ represents the number of nodes in the graph

In [ ]:
def EdgeDensity(graphs):
    edge_density_result = {}

    for name, g in graphs.items():  # Loop through each molecule graph
        num_nodes = len(g.nodes)
        num_edges = len(g.edges)

        # Avoid division by zero if graph has 0 or 1 nodes
        if num_nodes <= 1:
            density = 0.0
        else:
            density = (2 * num_edges) / (num_nodes * (num_nodes - 1))

        edge_density_result[name] = density  # Store result per molecule

    return edge_density_result

This version uses NetworkX’s built-in nx.density() function to calculate the edge density of a graph automatically.
It serves as a validation step to compare against the manually computed edge density and ensure correctness.

In [ ]:
def EdgeDensity_control(graphs):
    densities = {}
    for name, g in graphs.items():               # Loop through each graph
        densities[name] = nx.density(g)          # Use built-in NetworkX function
    return densities

## Wiener Index
The `wiener_index_bfs` function calculates the Wiener Index by summing the shortest-path distances between all pairs of nodes in a graph. 

It begins by initializing two variables: 
- total, which holds the cumulative distance
- counted, a set used to track pairs of nodes that have already been processed.

The function then loops through each node in the graph and uses Breadth-First Search (BFS) to compute the distances from that node to every other node. For each node, BFS explores all its neighbors and records their distances from the start node. Afterward, the function checks the computed distances, adding them to the total if they are valid (i.e., not the node itself and not already counted). To prevent double-counting, the function ensures that each pair of nodes is processed only once. 

Finally, after summing all valid distances, the function returns the Wiener Index, which represents the total sum of shortest-path distances between all unique node pairs in the graph.

In [ ]:
from collections import deque  # double-ended queue for BFS

def bfs_shortest_paths(graph, start):
    visited = {start: 0}           # Stores distances from start node (start node has distance 0)
    queue = deque([start])         # Initialize the BFS queue with the start node

    while queue:
        current = queue.popleft()  # Remove the current node from the front of the queue

        for neighbor in graph[current]:  # Iterate over neighbors of the current node
            if neighbor not in visited:  # If neighbor hasn't been visited yet
                visited[neighbor] = visited[current] + 1  # Set its distance (1 more than current)
                queue.append(neighbor)  # Add neighbor to queue for further exploration

    return visited  # Returns dictionary: {node: distance from start}

def wiener_index_bfs(graph):
    total = 0                      # Total sum of shortest-path distances
    counted = set()               # Keeps track of counted node pairs (unordered)

    for node in graph:            # Loop over each node in the graph
        distances = bfs_shortest_paths(graph, node)  # BFS distances from current node

        for other, dist in distances.items():        # Loop over all reachable nodes
            if node != other and (other, node) not in counted:
                total += dist                        # Add distance if pair not yet counted
                counted.add((node, other))           # Mark this pair as counted

    return total                 # Return the Wiener Index

### Final Calculation
The following is the final function used within the GUI. 

In [ ]:
def wiener_index(graphs):
    wiener_index_result = {}
    for name, g in graphs.items():
        adj = {node: list(g.neighbors(node)) for node in g.nodes}
        wiener_index_result[name] = wiener_index_bfs(adj)
    return wiener_index_result

This version uses NetworkX’s built-in nx.shortest_path_length() function to calculate the edge density of a graph automatically.
It serves as a validation step.

In [ ]:
def wiener_control(graphs):
    control_wiener_index = {}

    for name, g in graphs.items():
        total = 0
        nodes = list(g.nodes)
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                try:
                    dist = nx.shortest_path_length(g, nodes[i], nodes[j])
                    total += dist
                except nx.NetworkXNoPath:
                    pass  # skip if no path between nodes
        control_wiener_index[name] = total
    return control_wiener_index

## Petitjean
Using the radius and diameter, the **Petitjean Index** $ I $ is defined as:
$$
I = \frac{D - R}{R}
$$

### $ d(x,y) $

Is the shortest distance between node x and y. 

The function `distance_calc` calculates the d(x,y) between node $x$ and all other nodes $y$. We utilized the BFS algorithm to find the set of distances between node x and all other nodes. It returns the list of shortest paths.

In [ ]:
def distance_calc(node, graph):
    #bfs
    # intitialize structures
    visited = [node]
    queue = [node]
    distance = {node:0}
    
    while queue:
        m = queue.pop(0)
        d = distance[m]
        for neighbor in graph.neighbors(m):
            if neighbor not in visited:
                distance[neighbor] = d+1 # add to the distance of a neigbor since we are branching out
                visited.append(neighbor)
                queue.append(neighbor)
    return distance

### $E(x)$
The eccentricity of node $x$ is the maximum taken from the set d(x,y).

The function `E(graphs)` returns a list of eccentricity  for every node x.

In [ ]:
def Ecc(graphs):
    ecc = {}
    for name, g in graphs.items():
        g_no = remove_hydrogens(g) 
        ecc[name] = []
        for node in g_no.nodes:
            distance = distance_calc(node, g_no)
            ecc[name].append(max(distance.values()))
    return ecc

### Final Calculation
Where $R$ is the smallest eccentricity across all points and $D$ is the largest eccentricty.

In [ ]:
def petitjean(graphs):
    Ecc_dic = Ecc(graphs)
    petitJ = {}
    for name, ecc in Ecc_dic.items():
        D = max(ecc)
        R = min(ecc)
        I = (D - R)/R
        petitJ[name] = I
    return petitJ

This version uses NetworkX’s built-in nx.eccentricity() function to calculate the eccnetricity of all nodes of a graph automatically.
It serves as a validation step.

In [ ]:
def petitjean_control(graphs):
    petitJ = {}
    for name, g in graphs.items():
        G = remove_hydrogens(g) 
        ecc = nx.eccentricity(G)
        R = min(ecc.values())
        D = max(ecc.values())
        I = 0 if R == 0 else (D - R) / R
        petitJ[name] = I
    return petitJ

## Randić Index

Randić index is most widely applied in chemistry and pharmacology [source](https://www.sciencedirect.com/science/article/pii/S0972860017301408).
The Randić Index measures molecular branching based on atomic connectivity. It is defined as:

$$
\chi = \sum_{(v, w) \in E} \frac{1}{\sqrt{k_v \cdot k_w}}
$$

where $k_v$ and $k_w$ are the degrees of atoms $v$ and $w$ connected by a bond. While global descriptors like edge density, Wiener, and Petitjean indices capture overall shape and symmetry, the Randić Index complements them by providing sensitivity to local topology. 

For example, a linear chain and a branched molecule of comparable size may have similar global indices, but the Randić Index will be lower for the linear one due to its lower connectivity.


In [ ]:
# Takes the dictionary of graphs as input and outputs a dictionary of chi values
import math # import math for the square root

def randic(graphs):
    chi_all = {}
    for name, g in graphs.items():
        g_no = remove_hydrogens(g) 
        chi = 0
        for v, w in g_no.edges():
            kv = g_no.degree[v]
            kw = g_no.degree[w]
            if kv > 0 and kw > 0:
                chi += 1 / math.sqrt(kv * kw)
        chi_all[name] = chi
    return chi_all

# since there is no control function (see below), ensure the values are null
def randic_control(graphs):
    chi_all = {}
    for name, g in graphs.items():
        chi_all[name] = float('nan')
    return chi_all

### Validation code

Since we want to validate the index and we don't have control calculations of the indices, we will validate the overall performance using two examples:
- Hexane: should have a larger Randić since it's linear
- Neopentane: should be smaller since it's branched

In [ ]:
file_path = "validation_case.sdf" #fix if needed
with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()
validation_set = {"test.sdf": content}

graphs = create_graph_dic(validation_set)
ran_index = randic(graphs)
for name, chi in ran_index.items():
    print(name, ":", chi)

# Visualization

The `visualize_molecules_from_files()` function follows a series of steps to visualize molecular structures. 

First, it reads the molecular files (either SDF or MOL) and splits the data into individual lines. 

Then, it extracts metadata such as the number of atoms and bonds. The function builds a dictionary to store each atom's type and position using the extracted data, followed by creating a list of bonds that connect atoms. 

It then plots the molecule using matplotlib, drawing atoms as circles and bonds as lines, adjusting bond spacing for multiple bonds (like double bonds). Finally, the plot is displayed with equal axis scaling, and the title reflects the molecule's name, providing a clean and clear visualization of the molecule's structure.

In [ ]:
import matplotlib.pyplot as plt  # Importing the plotting library for creating visualizations

def visualize_molecules_from_files(files):
    # Get the list of mol structures by reading either SDF or MOL files
    mols = read_sdf(files) if is_sdf(files) else read_mol(files)
    figures = {}  # Dictionary to store fig objects

    # Loop through each molecule in the list of molecules
    for mol in mols:
        lines = split_record(mol)  # Split the molecule record into individual lines
        name = getName(mol)  # Extract the name or identifier of the molecule
        
        # Step 1: Extract metadata from the mol file
        prop = properties(lines)
        
        # Extract atom and bond lines based on the start and end indices
        atom_lines = lines[prop["ab_start"]:prop["ab_end"]]
        bond_lines = lines[prop["bb_start"]:prop["bb_end"]]
        
        # Step 2: Build the atom dictionary (storing atom types and coordinates)
        atoms = {}  # Initialize an empty dictionary to store atom data
        for i, line in enumerate(atom_lines):  # Iterate over each atom line
            # Extract the x and y coordinates of the atom from the line (first 10 characters for x, next 10 for y)
            x = float(line[0:10])
            y = float(line[10:20])
            atom_type = line[31:34].strip()  # Extract the atom type (e.g., 'C' for Carbon, 'O' for Oxygen)
            # Store the atom data in the dictionary with the atom index (i+1) as the key
            atoms[i + 1] = {'type': atom_type, 'pos': (x, y)}
        
        # Step 3: Build the bond list (storing bonds between atoms)
        bonds = []  # Initialize an empty list to store bond data
        for line in bond_lines:  # Iterate over each bond line
            # Extract atom indices (atom1 and atom2) and bond type from the line
            atom1 = int(line[0:3])
            atom2 = int(line[3:6])
            bond_type = int(line[6:9])
            # Store each bond as a tuple (atom1, atom2, bond_type) in the bonds list
            bonds.append((atom1, atom2, bond_type))
        
        # Step 4: Plotting the molecule
        fig, ax = plt.subplots(figsize=(6, 6))  

        for a1, a2, bond_type in bonds:  # Loop over each bond
            x1, y1 = atoms[a1]['pos']  # Get the coordinates of the first atom in the bond
            x2, y2 = atoms[a2]['pos']  # Get the coordinates of the second atom in the bond
            dx, dy = x2 - x1, y2 - y1  # Calculate the difference in x and y coordinates (direction of the bond)
            perp = (-dy, dx)  # Calculate a perpendicular vector to help with bond spacing
            norm = (perp[0]**2 + perp[1]**2)**0.5 or 1e-6  # Normalize the perpendicular vector to avoid division by zero
            offset = (perp[0]/norm * 0.12, perp[1]/norm * 0.12)  # Scale the offset for multiple bonds

            for i in range(bond_type):  # For multiple bonds (e.g., double bonds)
                shift = ((i - (bond_type - 1) / 2) * offset[0],  # Shift bonds to prevent overlap
                         (i - (bond_type - 1) / 2) * offset[1]) 
                # Plot the bond between atom1 and atom2, adjusting for bond type (single, double, etc.)
                ax.plot([x1 + shift[0], x2 + shift[0]], 
                         [y1 + shift[1], y2 + shift[1]],
                         color='gray', linewidth=2)  # Gray color and bond thickness set to 2
        
        # Draw atoms as circles
        for idx, info in atoms.items():  # Loop through each atom
            x, y = info['pos']  # Extract the atom's position (x, y)
            ax.plot(x, y, 'o', markersize=14, color='skyblue')  # Plot the atom as a circle with size 14
            ax.text(x, y, info['type'], fontsize=12, ha='center', va='center')  # Label the atom with its type
        
        # Final adjustments and display the plot
        ax.set_aspect("equal")  # Ensure equal scaling on both axes to maintain proportions
        ax.axis("off")  # Turn off the axis lines and labels for a cleaner view
        ax.set_title(f"Molecule: {name}")  # Set the plot title as the molecule's name

        # Save the figure under the molecule name in a way that is accessible to the GUI
        fig.tight_layout()
        
        out = widgets.Output()
        with out:
            display(fig)
        figures[name] = out

        plt.close(fig)  # Important to avoid duplicate plots
    return figures

# GUI Implementation

We first need to import the `ipywidget` library to create the widgets. The `ipyfilechooser` is needed for the file choosing interface. Finally, `IPython.display` is needed to display the created widgets. 

In [ ]:
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display
from IPython.display import clear_output

## Loading Function
This function allows for the loading of files or folders with the help of the `os` and `glob` libraries. We ensured the output is a dictionary of files.

In [ ]:
import os
import glob

def load_files(path):
    loaded = {}
    if os.path.isdir(path):
        for ext in ('*.mol', '*.sdf'):
            for file in glob.glob(os.path.join(path, ext)):
                with open(file) as f:
                    loaded[os.path.basename(file)] = f.read()
    elif os.path.isfile(path):
        with open(path) as f:
            loaded[os.path.basename(path)] = f.read()
    return loaded

## GUI Functions

### Set up Widgets

The GUI depends on the implementation of widgets. We created the widgets for the following:

- File Choosing: `fileType`, `file_picker`, `folder_picker`
- Checkboxes to choose descriptors: `descriptor_checkboxes`
- Run and upload buttons: `run_button`, `upload_button`
- Tab functionality: `file_tabs`, `tab_box`
- Loading Widget: `spinner`
- Output: `upload_output`, `output_box`

In [ ]:
fileType = widgets.ToggleButtons(
    options=['Single File', 'Folder (.mol)'],
    description='Input Type:',
    style={'description_width': 'initial'}
)

file_picker = FileChooser(os.getcwd())
file_picker.filter_pattern = ['*.mol', '*.sdf']
file_picker.use_dir_icons = True

folder_picker = FileChooser(os.getcwd())
folder_picker.show_only_dirs = True
folder_picker.use_dir_icons = True

descriptor_checkboxes = widgets.VBox([
    widgets.Label("Select Descriptor(s):"),
    widgets.Checkbox(value=False, description='Edge Density'),
    widgets.Checkbox(value=False, description='Wiener Index'),
    widgets.Checkbox(value=False, description='Petitjean Index'),
    widgets.Checkbox(value=False, description='Randić Index')
])

run_button = widgets.Button(
    description='Compute',
    button_style='success',
    icon='flask'
)

upload_button = widgets.Button(
    description='Upload',
    button_style='info',
    icon='upload'
)
upload_output = widgets.Output()

output_box = widgets.Output()

file_tabs = widgets.Tab()
file_tabs.layout = widgets.Layout(height='auto', width='100%')

tab_box = widgets.VBox([widgets.HTML("<b>Results by Molecule</b>"), file_tabs])
tab_box.layout.display = 'none'

spinner = widgets.HTML(
    value="<i class='fa fa-spinner fa-spin' style='font-size:24px; color:gray'></i> <b>Loading...</b>",
    layout=widgets.Layout(visibility='hidden')
)


Changes the GUI in real time (dynamicaly) if you toggle between folder and a file.

In [ ]:
# --- Dynamic input view ---
input_container = widgets.VBox()

def update_input_widget(change):
    input_container.children = [file_picker] if change['new'] == 'Single File' else [folder_picker]

fileType.observe(update_input_widget, names='value')
update_input_widget({'new': fileType.value})

### Functions

The `indices` function takes a graph as an input and uses the previous index functions to calculate them for all the molecules. It outputs a dictionary of the indices that will be accessed later on. 

The `reset` function clears the output boxes and tab output when needed.

In [ ]:
# Descriptor calculator
def indices(graphs):
    petitJ = (petitjean(graphs), petitjean_control(graphs))
    edgeDen = (EdgeDensity(graphs), EdgeDensity_control(graphs))
    wiener = (wiener_index(graphs), wiener_control(graphs))
    ran_ind = (randic(graphs), randic_control(graphs))


    all_Ind =  {"Petitjean Index": petitJ, "Wiener Index": wiener, "Edge Density": edgeDen, "Randić Index": ran_ind}

    return all_Ind

def reset():
    file_tabs.children = []
    tab_box.layout.display = 'none'
    with upload_output:
        clear_output()
    with output_box:
        clear_output()

### On button click
This section includes all the that will occur once the buttons are clicked!

`on_upload_clicked()`
- Runs the loading function describes above.
- Displays a message if it was succesful or not.
- Stores all files in a `files` disctionary that was declared global to  be used by all functions.

`on_compute_clicked`
- Once clicked initiated the spinner.
- Raises messages if certain criteria is not met (no descriptors clicked, no file selected).
- Iterates through every molecule and displays chosen indices by the user and the visualization of the graph.

In [ ]:
# --- Compute handler ---

def on_upload_clicked(b):
    main_app_box.layout.display = 'none'
    reset()

    # Global dictionary to store file contents
    global files
    path = file_picker.selected if fileType.value == 'Single File' else folder_picker.selected_path
    files = load_files(path)
    upload_output.clear_output()
    with upload_output:
        if not files:
            print("⚠️ No files loaded.")
            tab_box.layout.display = 'none'
        else:
            print(f"✅ Loaded {len(files)} file(s).")
            main_app_box.layout.display = 'flex'

def on_compute_clicked(b):
    spinner.layout.visibility = 'visible'
    reset()

    file_tabs.children = []  # ✅ clear previous tabs

    selected_descriptors = [
        cb.description for cb in descriptor_checkboxes.children[1:] if cb.value
    ]

    if not selected_descriptors:
        with upload_output:
            print("⚠️ Please select at least one descriptor.")
            spinner.layout.visibility = 'hidden'
        return

    if not files:
        with output_box:
            print( "⚠️ No files to compute.")
        tab_box.layout.display = 'none'
        spinner.layout.visibility = 'hidden'
        return
        
    tab_widgets = []
    #creates graphs and computes indices
    graphs = create_graph_dic(files)
    index_list = indices(graphs)
    #creates the plots for all molecules
    molecule_plots = visualize_molecules_from_files(files)
    
    for name, graph in graphs.items():
        desc_html = f"""
        <table style="border-collapse: collapse; width: 80%;">
            <thead>
                <tr>
                    <th style="text-align: left; border-bottom: 1px solid #ccc; padding: 6px;">Descriptor</th>
                    <th style="text-align: left; border-bottom: 1px solid #ccc; padding: 6px;">Value</th>
                    <th style="text-align: left; border-bottom: 1px solid #ccc; padding: 6px;">Control</th>
                </tr>
            </thead>
            <tbody>
        """
        for d in selected_descriptors:
            normal_val = index_list[d][0][name]
            control_val = index_list[d][1][name]
            desc_html += f"""
                <tr>
                    <td style="padding: 6px; border-bottom: 1px solid #eee;">{d}</td>
                    <td style="padding: 6px; border-bottom: 1px solid #eee;">{normal_val:.4f}</td>
                    <td style="padding: 6px; border-bottom: 1px solid #eee;">{control_val:.4f}</td>
                </tr>
            """
            
        desc_html += "</tbody></table>"
        desc_output = widgets.HTML(desc_html)

        
        vis_out = molecule_plots.get(name, widgets.HTML("<i>No visualization available</i>"))
            
        tab_widgets.append(widgets.VBox([
            widgets.HTML("<h3 style='color: #2c3e50; font-family: Arial; margin: 10px 0 5px; border-bottom: 2px solid #3498db;'>Indices</h3>"),
            desc_output,
            widgets.HTML("<h3 style='color: #2c3e50; font-family: Arial; margin: 10px 0 5px; border-bottom: 2px solid #2ecc71;'>Visualization</h3>"),
            vis_out
        ]))

    file_tabs.children = tab_widgets
    for i,name in enumerate(graphs.keys()):
        file_tabs.set_title(i, name)
        
    spinner.layout.visibility = 'hidden'
    with output_box:
        print( "✅ Computation complete. See tabs below.")
    
    tab_box.layout.display = 'block'

# associates button click to the functions
upload_button.on_click(on_upload_clicked)
run_button.on_click(on_compute_clicked)


### Final Layout
The following code organizes all the widgets on the GUI. 

We created the `main_app_box` to hold all the widgets that need to be shown after the upload button has been clicked. 

In [ ]:
main_app_box = widgets.VBox([
    descriptor_checkboxes,
    run_button,
    spinner,
    output_box,
    tab_box
])
main_app_box.layout.display = 'none'  # Hide initially
gui = widgets.VBox([
    fileType,
    input_container,
    upload_button,
    upload_output,
    main_app_box
])

# User Interface

In [ ]:
display(gui)